[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoAvancadoIME/blob/main/02_ajustamento_com_injuncoes.ipynb)

# Aula 2 - Ajustamento com Injunções

**Maj Diego - 1º Semestre / 2027**

**Objetivos:**

1. Interpretar injunções como informação adicional ou definição de datum.
2. Formular o modelo combinado com injunções.
3. Aplicar injunções ao modelo paramétrico e analisar seu efeito na solução.
4. Preparar a 1ª VE prática.

## O Problema

Considere que um primeiro ajustamento já foi concluído a partir das observações primitivas. Dele foram preservados:

- o vetor de parâmetros ajustados, $\hat X_0$;
- sua matriz de covariância, $\Sigma_{\hat X_0}$ (ou a matriz cofatora $Q_{\hat X_0}$ acompanhada do fator de variância);
- a definição do sistema de referência e das unidades.

Mais tarde, tornam-se disponíveis **novas observações**, coordenadas de controle ou relações que os parâmetros devem satisfazer. Refazer todo o ajustamento — remontando as equações normais a partir das observações primitivas — pode ser caro ou até impossível, pois os dados originais talvez não estejam mais disponíveis.

A ideia do ajustamento com injunções é aproveitar $\hat X_0$ como a melhor informação já consolidada e calcular somente a **correção** provocada pela nova informação:

$$
\hat X_{\text{novo}}=\hat X_0+\Delta X.
$$



![Esquema do ajustamento com injunções: solução anterior e covariância preservadas, incorporação de nova informação e correção dos parâmetros para obter a solução atualizada.](media/imgs/ajustamento_com_injuncoes.png)

**Figura — Atualização de um ajustamento por injunções.** A solução anterior e sua covariância são combinadas com a nova informação para calcular $\Delta X$. Na rede à direita, o traçado cinza representa a solução anterior e o azul, a atualizada; as setas destacam algumas correções. As elipses ilustram qualitativamente a incerteza das coordenadas. Esquema conceitual, sem escala.

## 1. Por que usar injunções?

Uma injunção acrescenta informação sobre os parâmetros. Ela pode cumprir funções diferentes:

1. **Definir o datum:** remover translações, rotações ou escala que não são determinadas pelas observações.
2. **Incorporar controle externo:** introduzir uma coordenada, distância, direção ou outro valor conhecido posteriormente.
3. **Representar uma relação física ou geométrica:** por exemplo, dois parâmetros iguais ou uma soma que deve permanecer constante.
4. **Atualizar um ajustamento existente:** combinar a solução anterior e sua covariância com novas informações.

Na forma linear, uma injunção é escrita como

$$
CX=W,
$$

em que cada linha de $C$ seleciona uma combinação dos parâmetros e $W$ contém o valor imposto.

- Uma injunção **forte** é tratada como exata. 
- Uma injunção **estocástica** reconhece que $W$ possui incerteza. 

> Essa escolha muda tanto a estimativa dos parâmetros quanto sua matriz de covariância.

## 2. Modelo paramétrico com injunções

No modelo paramétrico linear,

$$
L+v=AX,\qquad \min(v^TPv),
$$

as equações normais do ajustamento sem injunções são

$$
N\hat X=U,\qquad N=A^TPA,\qquad U=A^TPL.
$$

Ao impor a injunção forte $CX=W$, minimiza-se $v^TPv$ sujeito a essa igualdade. Com multiplicadores de Lagrange $K$, obtém-se

$$
\begin{bmatrix}
N&C^T\\
C&0
\end{bmatrix}
\begin{bmatrix}
\hat X\\K
\end{bmatrix}
=
\begin{bmatrix}
U\\W
\end{bmatrix}.
$$

O vetor $K$ mede a reação necessária para fazer a solução respeitar a restrição. Um multiplicador elevado pode indicar conflito entre a injunção e as observações, mas sua interpretação depende das unidades e dos pesos.

Quando o ajustamento anterior já está concluído, não é necessário reconstruir $N$ e $U$. Para uma injunção forte, basta usar $\hat X_0$ e $\Sigma_{\hat X_0}$:

$$
\hat X_c=\hat X_0+
\Sigma_{\hat X_0}C^T
\left(C\Sigma_{\hat X_0}C^T\right)^{-1}
\left(W-C\hat X_0\right),
$$

$$
\Sigma_{\hat X_c}=
\Sigma_{\hat X_0}-
\Sigma_{\hat X_0}C^T
\left(C\Sigma_{\hat X_0}C^T\right)^{-1}
C\Sigma_{\hat X_0}.
$$

A primeira expressão distribui a discrepância $W-C\hat X_0$ entre os parâmetros conforme suas correlações e incertezas. A segunda mostra que a incerteza é eliminada apenas na direção restringida; as demais direções podem conservar variância.

### Exemplo numérico: solução livre, injunção forte e injunção estocástica

O exemplo abaixo ajusta dois parâmetros, preserva a solução livre e sua covariância e, em seguida, introduz o controle sobre o primeiro parâmetro. A mesma informação é aplicada de duas formas para evidenciar a diferença entre considerar o controle exato ou incerto.

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# Ajustamento primitivo
A = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, 1.0]])
L = np.array([[100.2], [101.0], [0.9]])
P = np.eye(3)

N = A.T @ P @ A
U = A.T @ P @ L
x_livre = np.linalg.solve(N, U)
Sigma_livre = np.linalg.inv(N)  # variância de unidade de peso adotada como 1

# Nova informação: x_1 = 100
C = np.array([[1.0, 0.0]])
W = np.array([[100.0]])
inovacao = W - C @ x_livre

def atualizar(x0, Sigma0, C, W, Sigma_W=None):
    """Atualiza uma solução prévia com uma injunção forte ou estocástica."""
    if Sigma_W is None:
        Sigma_W = np.zeros((C.shape[0], C.shape[0]))
    Sigma_inovacao = C @ Sigma0 @ C.T + Sigma_W
    ganho = Sigma0 @ C.T @ np.linalg.inv(Sigma_inovacao)
    x_atualizado = x0 + ganho @ (W - C @ x0)
    Sigma_atualizada = Sigma0 - ganho @ C @ Sigma0
    return x_atualizado, Sigma_atualizada

x_forte, Sigma_forte = atualizar(x_livre, Sigma_livre, C, W)

# Controle com desvio-padrão de 0,20 unidade
Sigma_W = np.array([[0.20**2]])
x_estoc, Sigma_estoc = atualizar(x_livre, Sigma_livre, C, W, Sigma_W)

for nome, x, Sigma in [
    ("Livre", x_livre, Sigma_livre),
    ("Injunção forte", x_forte, Sigma_forte),
    ("Injunção estocástica", x_estoc, Sigma_estoc),
]:
    residuos_primitivos = A @ x - L
    print(f"{nome:22s} X = {x.ravel()}")
    print(f"{'':22s} C X - W = {(C @ x - W).item(): .6f}")
    print(f"{'':22s} resíduos = {residuos_primitivos.ravel()}")
    print(f"{'':22s} diag(Sigma_X) = {np.diag(Sigma)}\n")

# Verificações mínimas
assert np.allclose(C @ x_forte, W)
assert np.trace(Sigma_forte) <= np.trace(Sigma_estoc) <= np.trace(Sigma_livre)

Livre                  X = [100.166667 101.033333]
                       C X - W =  0.166667
                       resíduos = [-0.033333  0.033333 -0.033333]
                       diag(Sigma_X) = [0.666667 0.666667]

Injunção forte         X = [100.   100.95]
                       C X - W =  0.000000
                       resíduos = [-0.2  -0.05  0.05]
                       diag(Sigma_X) = [0.  0.5]

Injunção estocástica   X = [100.009434 100.954717]
                       C X - W =  0.009434
                       resíduos = [-0.190566 -0.045283  0.045283]
                       diag(Sigma_X) = [0.037736 0.509434]



### Como interpretar o exemplo

- A solução **livre** é o resultado do primeiro ajustamento e funciona como ponto de partida.
- A injunção **forte** faz $C\hat X=W$ exatamente. Por isso, a variância da combinação restringida torna-se nula.
- A injunção **estocástica** desloca a solução na direção do controle, mas mantém uma discrepância compatível com a incerteza atribuída a $W$.
- Os parâmetros não selecionados diretamente por $C$ também podem mudar quando estão correlacionados com o parâmetro restringido.
- Os resíduos das observações primitivas podem aumentar, pois a solução passa a conciliar essas observações com a informação adicional.

A redução da covariância não significa, por si só, que a injunção seja correta. Uma restrição incompatível também produz uma solução formalmente precisa. Por isso, devem ser analisados a inovação, os resíduos, o fator de variância e a procedência do controle.

## 3. Modelo combinado com injunções

No modelo combinado, observações e parâmetros aparecem na mesma relação funcional:

$$
F(L_a,X_a)=0.
$$

Linearizando em torno de valores aproximados $L_0$ e $X_0$, obtém-se

$$
Bv+A\Delta X+w=0,
$$

com

$$
B=\left.\frac{\partial F}{\partial L}\right|_{L_0,X_0},
\qquad
A=\left.\frac{\partial F}{\partial X}\right|_{L_0,X_0},
\qquad
w=F(L_0,X_0).
$$

As injunções são acrescentadas como equações sobre os parâmetros,

$$
G(X)=0
\quad\Longrightarrow\quad
C\Delta X+w_c=0,
$$

em que $C=\partial G/\partial X$ e $w_c=G(X_0)$. A separação entre $A$ e $B$ deve ser mantida: $A$ descreve variações dos parâmetros, enquanto $B$ descreve correções das observações.

O procedimento iterativo é:

1. escolher $L_0$ e $X_0$;
2. calcular $A$, $B$, $w$, $C$ e $w_c$;
3. resolver simultaneamente as correções $v$ e $\Delta X$;
4. atualizar $L_a=L_0+v$ e $X_a=X_0+\Delta X$;
5. repetir até que $\Delta X$ e o fechamento das equações sejam menores que as tolerâncias adotadas.

Injunções não lineares devem ser novamente linearizadas em cada iteração. Ao final, é necessário verificar tanto $F(L_a,X_a)\approx0$ quanto $G(X_a)\approx0$.

## 4. Injunção forte × estocástica

### Injunção forte

É usada quando a relação é considerada exata:

$$
CX=W.
$$

Ela elimina a variância na direção restringida e deve ser empregada com cautela. Exemplos típicos são uma definição matemática de datum ou uma identidade física assumida exata no modelo.

### Injunção estocástica

Quando o controle possui covariância $\Sigma_W$, ele é tratado como pseudo-observação:

$$
W=CX+\eta,\qquad \operatorname{Cov}(\eta)=\Sigma_W.
$$

A atualização da solução anterior torna-se

$$
\hat X_e=\hat X_0+
\Sigma_{\hat X_0}C^T
\left(C\Sigma_{\hat X_0}C^T+\Sigma_W\right)^{-1}
\left(W-C\hat X_0\right).
$$

Quanto menor $\Sigma_W$, mais a solução se aproxima da injunção forte; quanto maior $\Sigma_W$, menor é a influência do controle.

| Aspecto | Forte | Estocástica |
|---|---|---|
| Incerteza de $W$ | nula por hipótese | representada por $\Sigma_W$ |
| Atendimento a $CX=W$ | exato | aproximado e ponderado |
| Efeito na covariância | zera a variância na direção restringida | reduz, mas geralmente não zera |
| Uso recomendado | definição exata ou datum | controle medido ou informação externa incerta |

Em ambos os casos, documente origem, época, unidade, referencial e precisão da informação. Também verifique se $C\Sigma_{\hat X_0}C^T$ é invertível. Singularidade pode indicar injunções redundantes ou direções para as quais a solução anterior não fornece informação suficiente.

## 5. Verificações após a atualização

Uma solução atualizada deve ser acompanhada, no mínimo, das seguintes verificações:

1. **Fechamento da injunção:** avaliar $C\hat X-W$.
2. **Magnitude da inovação:** comparar $W-C\hat X_0$ com sua covariância
   $C\Sigma_{\hat X_0}C^T+\Sigma_W$.
3. **Resíduos:** verificar se a nova informação produz resíduos anormalmente grandes nas observações primitivas.
4. **Precisão:** comparar $\Sigma_{\hat X_0}$ e $\Sigma_{\hat X_{\text{novo}}}$, observando quais direções realmente ganharam informação.
5. **Posto e condicionamento:** identificar injunções redundantes, dependências lineares e sistemas numericamente instáveis.
6. **Consistência do modelo:** confirmar datum, unidades, época e eventuais correlações entre dados antigos e novos.

Para uma única inovação, o valor

$$
T=(W-C\hat X_0)^T
\left(C\Sigma_{\hat X_0}C^T+\Sigma_W\right)^{-1}
(W-C\hat X_0)
$$

é uma medida adimensional de incompatibilidade. Valores elevados sugerem que a nova informação, sua precisão ou o ajustamento anterior devem ser investigados antes da atualização definitiva.

## 1ª VE prática

**Tarefa:** ajustar uma pequena rede inicialmente sem injunções. Em seguida, conservar apenas $\hat X_0$ e $\Sigma_{\hat X_0}$ e introduzir uma nova informação de controle sem remontar as equações normais primitivas.

**Entregáveis:** notebook executável, formulação matricial, resultados comentados, verificações numéricas e conclusão técnica sobre o efeito de cada tipo de injunção.

### Rede proposta

```text
A -------- B
| \        |
|   \      |
|     \    |
D -------- C
```

**Enunciado:** ajuste a rede altimétrica formada pelos pontos A, B, C e D, usando as observações da tabela. Depois, incorpore o novo controle de altitude de D, primeiro como injunção forte e depois como injunção estocástica, e discuta seu efeito sobre a solução e as precisões.

| Observação | Grandeza observada | Valor (m) | Desvio-padrão (m) |
|---|---|---:|---:|
| 1 | Altitude de A: $H_A$ | 100,002 | 0,005 |
| 2 | Desnível A → B: $H_B-H_A$ | +45,253 | 0,003 |
| 3 | Desnível B → C: $H_C-H_B$ | −26,847 | 0,004 |
| 4 | Desnível C → D: $H_D-H_C$ | +53,618 | 0,003 |
| 5 | Desnível D → A: $H_A-H_D$ | −72,020 | 0,005 |
| 6 | Desnível A → C: $H_C-H_A$ | +18,409 | 0,004 |

Adote $\Delta h_{ij}=H_j-H_i$: o desnível é positivo quando o ponto de chegada está acima do ponto de partida. A altitude de A também foi observada e possui incerteza; portanto, **não fixe A como exato**.

Considere as seis observações independentes e seus desvios-padrão conhecidos. Use o vetor de incógnitas $X=[H_A\;H_B\;H_C\;H_D]^T$, o modelo $L+v=AX$ e a matriz $\Sigma_L=\operatorname{diag}(\sigma_1^2,\ldots,\sigma_6^2)$, na ordem da tabela. Adote $P=\Sigma_L^{-1}$ e $\Sigma_{\hat X_0}=(A^TPA)^{-1}$; não reescale essa covariância por um fator estimado dos resíduos.

Nesta atividade, **solução livre** significa solução sem a nova injunção em D. A observação de $H_A$, com peso finito, já determina a origem altimétrica e permite estimar as quatro altitudes. Uma rede composta somente por desníveis não teria essa origem definida.

**Novo controle, disponível somente após o primeiro ajustamento**

Uma determinação independente fornece $H_D^{\mathrm{controle}}=172{,}030\ \mathrm{m}$, com desvio-padrão $\sigma_D=0{,}004\ \mathrm{m}$. Essa informação é independente de todas as observações iniciais. Use $C=[0\;0\;0\;1]$ e $W=[172{,}030]\ \mathrm{m}$ em dois cenários, **ambos partindo da mesma solução inicial**:

- **Injunção forte:** imponha $H_D=172{,}030\ \mathrm{m}$ exatamente, desprezando a incerteza do controle apenas para essa comparação didática.
- **Injunção estocástica:** use o mesmo valor com $\Sigma_W=[(0{,}004)^2]\ \mathrm{m}^2$.

In [2]:
# Resolução da rede proposta para a 1ª VE prática
import numpy as np
from IPython.display import Markdown, display

def resolver_rede_ve():
    # Observações na ordem do enunciado; altitudes e desníveis em metros.
    A = np.array([
        [ 1,  0,  0,  0],
        [-1,  1,  0,  0],
        [ 0, -1,  1,  0],
        [ 0,  0, -1,  1],
        [ 1,  0,  0, -1],
        [-1,  0,  1,  0],
    ], dtype=float)
    L = np.array([100.002, 45.253, -26.847, 53.618, -72.020, 18.409])
    sigma_L = np.array([0.005, 0.003, 0.004, 0.003, 0.005, 0.004])
    P = np.diag(sigma_L**-2)
    N = A.T @ P @ A
    x0 = np.linalg.solve(N, A.T @ P @ L)
    Sigma0 = np.linalg.solve(N, np.eye(4))
    posto = np.linalg.matrix_rank(A)
    graus_liberdade = len(L) - posto

    # As atualizações recebem somente a solução inicial, sua covariância
    # e o novo controle. Não remontam as equações normais primitivas.
    C = np.array([[0., 0., 0., 1.]])
    W = np.array([172.030])
    sigma_controle = 0.004

    def atualizar_rede(x_anterior, Sigma_anterior, C, W, variancia_controle):
        inovacao = W - C @ x_anterior
        S = C @ Sigma_anterior @ C.T + np.array([[variancia_controle]])
        ganho = np.linalg.solve(S, C @ Sigma_anterior).T
        x_novo = x_anterior + ganho @ inovacao
        Sigma_nova = Sigma_anterior - ganho @ C @ Sigma_anterior
        Sigma_nova = (Sigma_nova + Sigma_nova.T) / 2
        return x_novo, Sigma_nova, inovacao.item(), S.item()

    xf, Sigmaf, inovacao, Sf = atualizar_rede(x0, Sigma0, C, W, 0.)
    xe, Sigmae, _, Se = atualizar_rede(x0, Sigma0, C, W, sigma_controle**2)
    casos = [("Inicial", x0, Sigma0), ("Forte", xf, Sigmaf), ("Estocástica", xe, Sigmae)]

    # Os dados primitivos são usados novamente apenas para diagnóstico.
    residuos = [A @ x - L for _, x, _ in casos]
    custos = [float(v @ P @ v) for v in residuos]
    residuos_controle = [float((C @ x - W).item()) for _, x, _ in casos]
    custo_controle = (residuos_controle[2] / sigma_controle)**2
    desvios = [np.sqrt(np.maximum(np.diag(Sigma), 0)) for _, _, Sigma in casos]
    Tf, Te = inovacao**2 / Sf, inovacao**2 / Se

    # Verificações de fechamento, covariância e custo da atualização.
    assert posto == 4 and graus_liberdade == 2
    assert np.allclose(C @ xf, W, rtol=0, atol=1e-10)
    assert np.allclose(C @ Sigmaf, 0, rtol=0, atol=1e-12)
    assert Sf > 0 and Se > 0
    assert np.linalg.matrix_rank(Sigmaf, tol=1e-12) == 3
    assert abs(residuos_controle[2]) < abs(inovacao)
    for _, _, Sigma in casos:
        assert np.allclose(Sigma, Sigma.T, rtol=0, atol=1e-12)
        assert np.linalg.eigvalsh(Sigma).min() >= -1e-12
    assert np.linalg.eigvalsh(Sigma0 - Sigmae).min() >= -1e-12
    assert np.linalg.eigvalsh(Sigmae - Sigmaf).min() >= -1e-12
    assert np.isclose(custos[1] - custos[0], Tf)
    assert np.isclose(custos[2] + custo_controle - custos[0], Te)

    def numero(valor, casas=6):
        return f"{valor:.{casas}f}".replace(".", ",")

    def matriz(valores, casas=6):
        linhas = [
            " & ".join(f"{v:.{casas}f}" for v in linha)
            for linha in np.atleast_2d(valores)
        ]
        return r"\begin{bmatrix}" + r" \\ ".join(linhas) + r"\end{bmatrix}"

    def tabela(cabecalhos, linhas):
        return "\n".join([
            "| " + " | ".join(cabecalhos) + " |",
            "| " + " | ".join(["---"] * len(cabecalhos)) + " |",
            *["| " + " | ".join(map(str, linha)) + " |" for linha in linhas],
        ])

    texto = [
        "### Resolução — rede da 1ª VE prática",
        "**1. Modelo e ajustamento inicial**",
        r"Adota-se $L+v=AX$, com $X=[H_A\;H_B\;H_C\;H_D]^T$.",
        "$$A=" + matriz(A, 0) + r",\qquad L=" + matriz(L[:, None], 3) + r"\ \mathrm{m}.$$",
        r"$$P=\operatorname{diag}(" + ", ".join(f"{p:.6f}" for p in np.diag(P))
        + r")\ \mathrm{m}^{-2}.$$",
        f"Posto de A: **{posto}**. Graus de liberdade iniciais: "
        f"**6 − {posto} = {graus_liberdade}**. "
        "A observação de A tem peso finito e determina a origem altimétrica.",
        r"Calculam-se $\hat X_0=N^{-1}A^TPL$ e $\Sigma_0=N^{-1}$, "
        r"com $N=A^TPA$. Os desvios-padrão fornecidos são conhecidos; "
        "não se aplica reescala pelas variâncias estimadas dos resíduos.",
        "**Solução inicial obtida com as seis observações:**",
        r"$$\hat X_0=" + matriz(x0[:, None], 6) + r"\ \mathrm{m},"
        r"\qquad \text{ordem: A, B, C, D}.$$",
        "A altitude ajustada de A coincide com sua observação porque ela é a única "
        "observação absoluta da rede: os desníveis determinam as diferenças "
        "entre as altitudes, e essa observação determina a origem. "
        "Sua incerteza permanece igual a 5 mm.",
        tabela(
            ["Ponto", "Altitude inicial ajustada (m)", "Desvio-padrão (mm)"],
            [[ponto, numero(x0[i]), numero(desvios[0][i]*1000, 3)]
             for i, ponto in enumerate("ABCD")],
        ),
        r"A matriz de covariância que será levada ao item 2 é "
        r"$\Sigma_0=(A^TPA)^{-1}$. Em mm²:",
        r"$$\Sigma_0=" + matriz(Sigma0*1e6) + r"\ \mathrm{mm}^2.$$",
        r"Os resíduos iniciais, na ordem das seis observações, são:",
        r"$$v_0=A\hat X_0-L=" + matriz(residuos[0][:, None]*1000)
        + r"\ \mathrm{mm}.$$",
        f"A soma ponderada inicial é **v₀ᵀPv₀ = {numero(custos[0])}**. "
        "Com isso, o ajustamento inicial está concluído. "
        "O novo controle em D só é incorporado na etapa seguinte.",
        "**2. Atualização a partir da solução inicial**",
        r"Em ambos os cenários, $d=W-C\hat X_0$, "
        r"$S=C\Sigma_0C^T+\Sigma_W$, $G=\Sigma_0C^TS^{-1}$, "
        r"$\hat X=\hat X_0+Gd$ e $\Sigma=\Sigma_0-GC\Sigma_0$.",
        f"A inovação comum é **{numero(1000*inovacao, 6)} mm**. "
        "Na injunção forte, a variância do controle é zero por hipótese; "
        "na estocástica, é 16 mm².",
        tabela(
            ["Cenário", "S (mm²)", "σ da inovação (mm)", "d/σ", "T = d²/S"],
            [[nome, numero(S*1e6), numero(np.sqrt(S)*1e3),
              numero(inovacao/np.sqrt(S)), numero(inovacao**2/S)]
             for nome, S in [("Forte", Sf), ("Estocástica", Se)]],
        ),
        "**3. Altitudes, precisões e alterações**",
        tabela(
            ["Ponto", "Inicial (m)", "Forte (m)", "Estocástica (m)",
             "σ inicial (mm)", "σ forte (mm)", "σ estocástica (mm)"],
            [[ponto, numero(x0[i]), numero(xf[i]), numero(xe[i]),
              *[numero(s[i]*1000, 3) for s in desvios]]
             for i, ponto in enumerate("ABCD")],
        ),
        tabela(
            ["Ponto", "Correção forte (mm)", "Correção estocástica (mm)"],
            [[ponto, numero((xf[i]-x0[i])*1000), numero((xe[i]-x0[i])*1000)]
             for i, ponto in enumerate("ABCD")],
        ),
        "Matrizes de covariância na ordem A, B, C, D, em **mm²** "
        "(multiplique por 10⁻⁶ para obter m²):",
    ]
    for nome, _, Sigma in casos:
        texto.extend([f"**{nome}:**", "$$" + matriz(Sigma*1e6) + "$$"])
    texto.extend([
        "**4. Resíduos e somas ponderadas**",
        r"Os resíduos seguem $v=A\hat X-L$; o resíduo do controle segue $C\hat X-W$.",
        tabela(
            ["Observação", "Inicial (mm)", "Forte (mm)", "Estocástica (mm)"],
            [[rotulo, *[numero(v[i]*1000) for v in residuos]]
             for i, rotulo in enumerate(["H_A", "A → B", "B → C", "C → D", "D → A", "A → C"])],
        ),
        tabela(
            ["Solução", "vᵀPv (primitivas)", "C X − W (mm)"],
            [[nome, numero(custos[i]), numero(residuos_controle[i]*1000)]
             for i, (nome, _, _) in enumerate(casos)],
        ),
        "Na linha inicial, a discrepância com o controle é apenas diagnóstica: "
        "ele ainda não entrou no ajustamento. Na injunção forte, "
        "a igualdade é exata e não recebe um peso finito.",
        f"Na versão estocástica, a contribuição ponderada do controle é "
        f"**{numero(custo_controle)}** e a soma total, incluindo as observações "
        f"primitivas, é **{numero(custos[2]+custo_controle)}**. "
        "Essas somas são adimensionais. O aumento do custo das observações "
        "primitivas representa a conciliação com a nova informação.",
        "**5. Condicionamento e interpretação**",
        f"O número de condição de N na norma 2 é **{numero(np.linalg.cond(N), 3)}**, "
        "sem sinal de mau condicionamento relevante neste exemplo. "
        "As duas covariâncias da inovação são escalares positivas, portanto invertíveis.",
        "A covariância da solução forte tem posto 3: a altitude de D é exata "
        "por hipótese e sua linha e coluna têm covariância nula. Essa singularidade "
        "é esperada; a atualização inverte S, não a covariância final.",
        "A inovação é menor que um desvio-padrão em ambos os cenários "
        f"(T forte = {numero(Tf, 3)}; T estocástica = {numero(Te, 3)}). "
        "Não há discrepância acentuada em relação às precisões adotadas. "
        "Isso apoia a compatibilidade do controle neste exercício, sem comprovar "
        "a ausência de erros sistemáticos ou de problemas de referencial.",
        "As altitudes de A, B e C também mudam porque suas covariâncias com D "
        "são diferentes de zero. A injunção forte elimina a incerteza em D "
        "por hipótese; a estocástica mantém incerteza e uma discrepância residual "
        "com o controle. **A versão estocástica representa a precisão informada "
        "de 4 mm para o novo controle.**",
        "**Verificações numéricas:** fechamento da injunção forte, simetria e "
        "semidefinição positiva das covariâncias, redução de incerteza e "
        "identidades dos custos da atualização conferidos.",
    ])
    display(Markdown("\n\n".join(texto)))

resolver_rede_ve()


### Resolução — rede da 1ª VE prática

**1. Modelo e ajustamento inicial**

Adota-se $L+v=AX$, com $X=[H_A\;H_B\;H_C\;H_D]^T$.

$$A=\begin{bmatrix}1 & 0 & 0 & 0 \\ -1 & 1 & 0 & 0 \\ 0 & -1 & 1 & 0 \\ 0 & 0 & -1 & 1 \\ 1 & 0 & 0 & -1 \\ -1 & 0 & 1 & 0\end{bmatrix},\qquad L=\begin{bmatrix}100.002 \\ 45.253 \\ -26.847 \\ 53.618 \\ -72.020 \\ 18.409\end{bmatrix}\ \mathrm{m}.$$

$$P=\operatorname{diag}(40000.000000, 111111.111111, 62500.000000, 111111.111111, 40000.000000, 62500.000000)\ \mathrm{m}^{-2}.$$

Posto de A: **4**. Graus de liberdade iniciais: **6 − 4 = 2**. A observação de A tem peso finito e determina a origem altimétrica.

Calculam-se $\hat X_0=N^{-1}A^TPL$ e $\Sigma_0=N^{-1}$, com $N=A^TPA$. Os desvios-padrão fornecidos são conhecidos; não se aplica reescala pelas variâncias estimadas dos resíduos.

**Solução inicial obtida com as seis observações:**

$$\hat X_0=\begin{bmatrix}100.002000 \\ 145.255191 \\ 118.408530 \\ 172.025331\end{bmatrix}\ \mathrm{m},\qquad \text{ordem: A, B, C, D}.$$

A altitude ajustada de A coincide com sua observação porque ela é a única observação absoluta da rede: os desníveis determinam as diferenças entre as altitudes, e essa observação determina a origem. Sua incerteza permanece igual a 5 mm.

| Ponto | Altitude inicial ajustada (m) | Desvio-padrão (mm) |
| --- | --- | --- |
| A | 100,002000 | 5,000 |
| B | 145,255191 | 5,634 |
| C | 118,408530 | 5,708 |
| D | 172,025331 | 5,976 |

A matriz de covariância que será levada ao item 2 é $\Sigma_0=(A^TPA)^{-1}$. Em mm²:

$$\Sigma_0=\begin{bmatrix}25.000000 & 25.000000 & 25.000000 & 25.000000 \\ 25.000000 & 31.742475 & 27.729097 & 27.006689 \\ 25.000000 & 27.729097 & 32.580825 & 30.574136 \\ 25.000000 & 27.006689 & 30.574136 & 35.716276\end{bmatrix}\ \mathrm{mm}^2.$$

Os resíduos iniciais, na ordem das seis observações, são:

$$v_0=A\hat X_0-L=\begin{bmatrix}-0.000000 \\ 0.190635 \\ 0.338907 \\ -1.198997 \\ -3.330546 \\ -2.470457\end{bmatrix}\ \mathrm{mm}.$$

A soma ponderada inicial é **v₀ᵀPv₀ = 0,996098**. Com isso, o ajustamento inicial está concluído. O novo controle em D só é incorporado na etapa seguinte.

**2. Atualização a partir da solução inicial**

Em ambos os cenários, $d=W-C\hat X_0$, $S=C\Sigma_0C^T+\Sigma_W$, $G=\Sigma_0C^TS^{-1}$, $\hat X=\hat X_0+Gd$ e $\Sigma=\Sigma_0-GC\Sigma_0$.

A inovação comum é **4,669454 mm**. Na injunção forte, a variância do controle é zero por hipótese; na estocástica, é 16 mm².

| Cenário | S (mm²) | σ da inovação (mm) | d/σ | T = d²/S |
| --- | --- | --- | --- | --- |
| Forte | 35,716276 | 5,976310 | 0,781327 | 0,610472 |
| Estocástica | 51,716276 | 7,191403 | 0,649311 | 0,421604 |

**3. Altitudes, precisões e alterações**

| Ponto | Inicial (m) | Forte (m) | Estocástica (m) | σ inicial (mm) | σ forte (mm) | σ estocástica (mm) |
| --- | --- | --- | --- | --- | --- | --- |
| A | 100,002000 | 100,005268 | 100,004257 | 5,000 | 2,739 | 3,594 |
| B | 145,255191 | 145,258721 | 145,257629 | 5,634 | 3,365 | 4,200 |
| C | 118,408530 | 118,412527 | 118,411290 | 5,708 | 2,532 | 3,809 |
| D | 172,025331 | 172,030000 | 172,028555 | 5,976 | 0,000 | 3,324 |

| Ponto | Correção forte (mm) | Correção estocástica (mm) |
| --- | --- | --- |
| A | 3,268435 | 2,257246 |
| B | 3,530785 | 2,438429 |
| C | 3,997184 | 2,760533 |
| D | 4,669454 | 3,224816 |

Matrizes de covariância na ordem A, B, C, D, em **mm²** (multiplique por 10⁻⁶ para obter m²):

**Inicial:**

$$\begin{bmatrix}25.000000 & 25.000000 & 25.000000 & 25.000000 \\ 25.000000 & 31.742475 & 27.729097 & 27.006689 \\ 25.000000 & 27.729097 & 32.580825 & 30.574136 \\ 25.000000 & 27.006689 & 30.574136 & 35.716276\end{bmatrix}$$

**Forte:**

$$\begin{bmatrix}7.500975 & 6.096371 & 3.599298 & 0.000000 \\ 6.096371 & 11.321498 & 4.610613 & 0.000000 \\ 3.599298 & 4.610613 & 6.408506 & 0.000000 \\ 0.000000 & 0.000000 & 0.000000 & 0.000000\end{bmatrix}$$

**Estocástica:**

$$\begin{bmatrix}12.914830 & 11.944783 & 10.220255 & 7.734509 \\ 11.944783 & 17.639347 & 11.763017 & 8.355339 \\ 10.220255 & 11.763017 & 14.505707 & 9.459037 \\ 7.734509 & 8.355339 & 9.459037 & 11.049914\end{bmatrix}$$

**4. Resíduos e somas ponderadas**

Os resíduos seguem $v=A\hat X-L$; o resíduo do controle segue $C\hat X-W$.

| Observação | Inicial (mm) | Forte (mm) | Estocástica (mm) |
| --- | --- | --- | --- |
| H_A | -0,000000 | 3,268435 | 2,257246 |
| A → B | 0,190635 | 0,452985 | 0,371819 |
| B → C | 0,338907 | 0,805306 | 0,661012 |
| C → D | -1,198997 | -0,526726 | -0,734714 |
| D → A | -3,330546 | -4,731565 | -4,298117 |
| A → C | -2,470457 | -1,741709 | -1,967169 |

| Solução | vᵀPv (primitivas) | C X − W (mm) |
| --- | --- | --- |
| Inicial | 0,996098 | -4,669454 |
| Forte | 1,606570 | 0,000000 |
| Estocástica | 1,287266 | -1,444637 |

Na linha inicial, a discrepância com o controle é apenas diagnóstica: ele ainda não entrou no ajustamento. Na injunção forte, a igualdade é exata e não recebe um peso finito.

Na versão estocástica, a contribuição ponderada do controle é **0,130436** e a soma total, incluindo as observações primitivas, é **1,417702**. Essas somas são adimensionais. O aumento do custo das observações primitivas representa a conciliação com a nova informação.

**5. Condicionamento e interpretação**

O número de condição de N na norma 2 é **37,833**, sem sinal de mau condicionamento relevante neste exemplo. As duas covariâncias da inovação são escalares positivas, portanto invertíveis.

A covariância da solução forte tem posto 3: a altitude de D é exata por hipótese e sua linha e coluna têm covariância nula. Essa singularidade é esperada; a atualização inverte S, não a covariância final.

A inovação é menor que um desvio-padrão em ambos os cenários (T forte = 0,610; T estocástica = 0,422). Não há discrepância acentuada em relação às precisões adotadas. Isso apoia a compatibilidade do controle neste exercício, sem comprovar a ausência de erros sistemáticos ou de problemas de referencial.

As altitudes de A, B e C também mudam porque suas covariâncias com D são diferentes de zero. A injunção forte elimina a incerteza em D por hipótese; a estocástica mantém incerteza e uma discrepância residual com o controle. **A versão estocástica representa a precisão informada de 4 mm para o novo controle.**

**Verificações numéricas:** fechamento da injunção forte, simetria e semidefinição positiva das covariâncias, redução de incerteza e identidades dos custos da atualização conferidos.